# The 755 doubly-labelled laser images (T1, T2)

`p2-tests.md` T1 and T2. Extract: `sql/extract_laser_annotations.sql` on the
2026-09-25 nightly backup, frozen to `data/laser_pairs/` for the 755 images (numeric
labeller ids only). T1 asks whether the pairs are independent; T2 was to measure their
spatial agreement.

**Result: they are not independent, so T2 cannot run on them.** What they measure
instead is how labellers treat a pre-annotation.

In [1]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.precision", 2)

from fishsense_cscw import laser_pairs as lp
a = lp.load_annotations()
print("reproduced pair images:", len(lp.pair_images(a)), "(effort study: 755)")
p = lp.pairs(a)
print(f"pairs with a dot on both sides: {len(p)};  from two different LS projects: {(~p.same_project).sum()};  "
      f"gap median {p.gap_days.median():.0f} days")

reproduced pair images: 755 (effort study: 755)


pairs with a dot on both sides: 717;  from two different LS projects: 715;  gap median 261 days


## What the second labeller did, and how far their label sits from the first

In [2]:
p.groupby(["canvas", "later_action"]).distance_px.agg(
    pairs="size", exact_copies=lambda d: int((d < 1e-6).sum()),
    p50="median", p90=lambda d: d.quantile(.9), max="max").round(2)

pairs  exact_copies     p50     p90     max
canvas     later_action                                             
3987->4014 accepted        448           448    0.00    0.00    0.00
           moved            11             0    2.87   15.21  355.25
           none              4             0  389.76  795.34  852.39
4014->4014 accepted        178             2    0.41    0.54    3.08
           moved            74             0    0.92    2.55    5.61
           none              2             0    0.53    0.56    0.57

- **accepted:** the later project was pre-annotated with the earlier *human* label, and
  the labeller submitted it unchanged. All 448 mixed-canvas accepts are exact to float
  precision, so the 3987- and 4014-wide canvases share a pixel frame.
- **moved:** nudged the seed, median under 1 px.
- **none:** drawn with no seed. Only six, and four disagree by hundreds of pixels
  (different dots), so they are not placement noise either.

The seeds were the earlier human labels, not detector output: where a v2 prediction
exists for these images, the accepted seed is 0.0–0.4 px from the old human label and
1.2–1.4 px from the detector.

In [3]:
import numpy as np
P = pd.read_csv(lp.PAIRS / "laserprediction.csv").sort_values("created_at").groupby("image_id").last()
m = p.merge(P[["x", "y"]], left_on="image_id", right_index=True, how="inner")
m["seed_to_detector"] = np.hypot(m.later_x - m.x, m.later_y - m.y)
m["seed_to_first_human"] = m.distance_px
m[m.later_action == "accepted"].groupby("canvas")[["seed_to_first_human", "seed_to_detector"]].median().round(2)

,seed_to_first_human,seed_to_detector
canvas,,
3987->4014,0.00,1.42
4014->4014,0.43,1.17


## What the pairs *do* measure: anchoring on pre-annotations

Split by whether the seed's source label is now superseded. The flag has no recorded
reason (validator outlier or routine replacement), so the second table asks whether
anything live replaced the accepted copy.

In [4]:
lp.anchoring(p).round(3)

later_action,accepted,moved,accept_rate
first_superseded,,,
seed's source live,437,28,0.94
seed's source now superseded,189,57,0.77


In [5]:
acc_bad = p[(p.later_action == "accepted") & p.first_superseded]
d = lp.live_label_distance(acc_bad)
print(f"accepted copies of a now-superseded label: {len(acc_bad)}")
print(f"  image has no live laser label at all now : {int(d.isna().sum())}")
print(f"  a live label within 5 px                 : {int((d <= 5).sum())}")
print(f"  a live label moved >5 px (a correction)   : {int((d > 5).sum())}")

accepted copies of a now-superseded label: 189
  image has no live laser label at all now : 184
  a live label within 5 px                 : 5
  a live label moved >5 px (a correction)   : 0


## Result

**T1: there is no laser inter-annotator ceiling in prod.** 626 of 717 pairs are
accepted copies of a seeded earlier label and 85 are sub-pixel nudges of one. That leaves
6 unseeded pairs, too few, and 4 of them are about *which* dot. `HANDOFF.md` §4.2 was right
that the ceiling does not exist; `p2-inventory.md` §3.1 was wrong to say it did. The laser
dot needs its own collection, which is cheap: laser labelling runs a median 12.9 s per
frame (`MEASUREMENTS.md`). Fold it into T21.

**T2 is replaced by an anchoring result.**
- Shown a seed, labellers **accept it unchanged 87 % of the time** (626 / 711).
- When the seed's source is now superseded, they still accept it **77 %** of the time,
  against **94 %** for live seeds, so they discriminate, but weakly.
- When they move a seed the median move is under 1 px, which does not fix a wrong dot.
- **184 images** hold only accepted copies of a superseded label and now have **no
  live laser label**. Not one accepted seed was later corrected by more than 5 px.

Why this matters for P2:
1. **"93 % byte-exact agreement" with the detector** (`auto_accept.py`) is an
   *acceptance* rate for the same interface. It cannot be read as accuracy.
2. **Evaluate detectors only against labels drawn without a seed** (`origin == "manual"`,
   no `parent_prediction`), or agreement is circular.
3. It is the observational baseline for **T22**: the pre-annotation trial has to measure
   both speed and anchoring, because the corpus already shows the anchoring.

Caveats:
- "Superseded" is the line-fit validator's judgement, not ground truth.
- The data cannot say whether a seed was flagged before or after it was accepted.
- The seeds here were prior *human* labels. Anchoring on *model* seeds is plausible but
  not measured here.